# Logits Preprocessing and Data Engineering

In [1]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/data/extension/mitigation/datasets',
            'current': 'base', # 'base' or 'prompted',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'output_path' : '/workspaces/CodeSmells/datax/code_smells/logits/mitigation',
        'callbacks_path' : '/workspaces/CodeSmells/datax/code_smells/callbacks/mitigation',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [3]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/data_en.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Dataset

In [8]:
print(f"{params['dataset']['path']}/{params['dataset']['current']}.json")

/workspaces/CodeSmells/data/extension/mitigation/datasets/base.json


In [9]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['current']}.json", )

In [10]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
0,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,def get_uploaded_datasets(self):\n if l...,https://github.com/OpenMined/PySyft.git,Python,...,260,21,W0719,9,12,9,100,"raise Exception(""User cannot connect to this d...",Warning,361
1,115917,6eb408a9973fbc24c973d6524dc34cb9b1e0ee05,mindsdb,mindsdb/api/mongo/responders/delete.py,delete.py,_result,del model interface,"def _result(self, query, request_env, mindsdb_...",https://github.com/mindsdb/mindsdb.git,Python,...,253,18,W0719,10,12,10,129,"raise Exception(""For db.predictors.delete oper...",Warning,293
2,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,def get_uploaded_datasets(self):\n if l...,https://github.com/OpenMined/PySyft.git,Python,...,260,21,W0719,6,12,6,190,"raise Exception(""Either proxy not running or n...",Warning,361
3,116232,b734716c29a8b7613354e160ea1a3aea111f0831,mindsdb,mindsdb/interfaces/database/integrations.py,integrations.py,get_handler,Improve handlers files storage,"def get_handler(self, name, company_id=None, c...",https://github.com/mindsdb/mindsdb.git,Python,...,457,45,W0719,15,12,15,97,"raise Exception(f""Cant find handler for '{inte...",Warning,536
4,115111,d725c063e3ac3ab919fc8ed5f969fef8bc478209,mindsdb,mindsdb/api/mysql/mysql_proxy/datahub/datanode...,integration_datanode.py,select,fix,"def select(self, query):\n result = sel...",https://github.com/mindsdb/mindsdb.git,Python,...,141,21,W0719,4,12,4,49,raise Exception(result.error_message),Warning,148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,115229,4f2861b6ded274d4a41322c107ace8107e86ebea,mindsdb,mindsdb/interfaces/database/views.py,views.py,add,store integration in sql of view (before save it),"def add(self, name, query, integration_name, c...",https://github.com/mindsdb/mindsdb.git,Python,...,247,25,W0719,21,16,21,88,"raise Exception(f""Can't find integration with ...",Warning,264
496,282637,fae93c67adc9015c1466712f9c8ffa35a8b70872,OpenBBTerminal,bots/economy/usbonds.py,usbonds.py,usbonds_command,Refactor Bot (#1326)\n\n* First commit\r\n\r\n...,def usbonds_command():\n \n\n # Debug us...,https://github.com/OpenBB-finance/OpenBBTermin...,Python,...,469,45,W0719,12,8,12,50,"raise Exception(""No available data found"")",Warning,545
497,2842,ccd4b9330a186090cc87e94d2da1093d45de329f,PySyft,packages/syft/src/syft/oblv/model.py,model.py,request_publish,Changes for model,"def request_publish(self, dataset_id, sigma = ...",https://github.com/OpenMined/PySyft.git,Python,...,383,31,W0719,2,12,2,116,"raise Exception(""No Domain Clients added. Set ...",Warning,497
498,116308,326622a6fb33664de21ee1627f5f083f11b59a9e,mindsdb,mindsdb/integrations/handlers/ludwig_handler/l...,ludwig_handler.py,_learn,fix: add hyperopt,"def _learn(self, statement):\n model_na...",https://github.com/mindsdb/mindsdb.git,Python,...,374,50,W0719,8,12,8,85,"raise Exception(""Ludwig handler does not suppo...",Warning,466


#### Model Loading

In [11]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

2025-07-02 15:40:31.206182: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751470831.222408  992180 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751470831.228122  992180 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-02 15:40:31.248318: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [13]:
print(model.config)
print(model.__class__)
print(tokenizer.__class__)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "float32",
  "transformers_version": "4.52.4",
  "use_cache": true,
  "vocab_size": 32016
}

<class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
<class 'transformers.models.code_llama.tokenization_code_llama_fast.CodeLlamaTokenizerFast'>


#### preprocess dataset

In [14]:
df_dataset['input_ids'] = df_dataset[params['dataset']['content_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
if params['dataset']['current'] == 'prompted':
    df_dataset['prompt_ids'] = df_dataset[params['dataset']['prompt_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
df_dataset['input_lenght'] = df_dataset['input_ids'].map(lambda input_ids: len(input_ids))

#### Softmax Normalization and Data Engineering

In [15]:
def topk_tuple(logit_vocab_tensor, largest, tokenizer_fn):
    """
    Return the decoded top-1 (or bottom-1) token and its logit value.
    """
    topk = logit_vocab_tensor.topk(k=1, largest=largest)
    top_token_id = topk.indices[0].item()
    decoded_token = tokenizer_fn.decode([top_token_id])  # Already handles special tokens and spaces
    return (decoded_token, topk.values[0].item())

In [16]:
def analyze_logits(logit_tensor_sequence, input_token_ids, tokenizer_fn, skip_first_token=True):
    """
    Analyze logits for each token in a sequence.

    Args:
        logit_tensor_sequence (List[Tensor]): List of vocab-sized logits for each token position.
        input_token_ids (List[int] or Tensor): Token IDs of the input prompt.
        tokenizer_fn: HuggingFace tokenizer with .decode() method.
        skip_first_token (bool): Whether to skip the first token prediction (default: True).

    Returns:
        dict with:
            - "max_cases": list of (decoded top-1 token, logit value)
            - "min_cases": list of (decoded bottom-1 token, logit value)
            - "actual_logits": list of (decoded ground-truth token, logit value)
    """
    max_cases = []
    min_cases = []
    actual_logits = []

    start_index = 1 if skip_first_token else 0
    token_targets = input_token_ids[start_index:]

    for position, token_id in enumerate(token_targets):
        vocab_logits = logit_tensor_sequence[position]

        # Top-1 max and min predictions
        max_case = topk_tuple(logit_vocab_tensor=vocab_logits, largest=True, tokenizer_fn=tokenizer_fn)
        min_case = topk_tuple(logit_vocab_tensor=vocab_logits, largest=False, tokenizer_fn=tokenizer_fn)

        # Actual token logit
        decoded_token = tokenizer_fn.decode([int(token_id)])
        logit_value = vocab_logits[int(token_id)].item()
        actual_case = (decoded_token, logit_value)

        max_cases.append(max_case)
        min_cases.append(min_case)
        actual_logits.append(actual_case)

    return {
        "max_cases": max_cases,
        "min_cases": min_cases,
        "actual_logits": actual_logits
    }

In [17]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [18]:
callbacks_dir = f"{params['callbacks_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
out = np.load(f"{callbacks_dir}/logits_tensor[0]_batch[0].npy")

print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 361, 32016)


In [19]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

logit_dict = analyze_logits(
    logit_tensor_sequence = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    input_token_ids = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
)

In [20]:
assert len(set(len(v) for v in logit_dict.values())) == 1, "All key array values in logit_dict do not have the same length"

#### Processing all the Batches

In [21]:
def process_logit_batches(tokenizer, tokenized_inputs, num_samples=10000, skip_first_token=True):
    """
    Process multiple saved logits files and extract:
    - top-1 max logit predictions,
    - top-1 min logit predictions,
    - actual logits for ground-truth tokens.

    Args:
        tokenizer: HuggingFace tokenizer instance.
        tokenized_inputs (List[Tensor]): Tokenized input prompts (one per sample).
        num_samples (int): Number of samples to process.
        skip_first_token (bool): Whether to skip the first token prediction.

    Returns:
        Tuple of lists: (max_logit_predictions, min_logit_predictions, actual_logits)
    """
    max_logit_predictions = []
    min_logit_predictions = []
    actual_logit_scores = []

    softmax_fn = torch.nn.Softmax(dim=0)
    base_path = f"{params['callbacks_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"

    for sample_idx in range(num_samples):
        logits_file_path = f"{base_path}/logits_tensor[{sample_idx}]_batch[{sample_idx}].npy"
        logits_array = np.load(logits_file_path)[0]  # Shape: [sequence_length, vocab_size]

        # Apply softmax to each token’s logits
        normalized_logits = [softmax_fn(torch.from_numpy(token_logits)) for token_logits in logits_array]

        # Analyze logits using the unified function
        result = analyze_logits(
            logit_tensor_sequence=normalized_logits,
            input_token_ids=tokenized_inputs[sample_idx],
            tokenizer_fn=tokenizer,
            skip_first_token=skip_first_token
        )

        max_logit_predictions.append(result["max_cases"])
        min_logit_predictions.append(result["min_cases"])
        actual_logit_scores.append(result["actual_logits"])

        logging.info(f"Processed sample {sample_idx}")
        print(f"Processed sample {sample_idx}")

    return max_logit_predictions, min_logit_predictions, actual_logit_scores

In [22]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [23]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = process_logit_batches(
    tokenizer=tokenizer , tokenized_inputs=input_ids_list, 
    num_samples = len(df_dataset)
) #<---WARNING TIME Consuming

Processed sample 0
Processed sample 1
Processed sample 2
Processed sample 3
Processed sample 4
Processed sample 5
Processed sample 6
Processed sample 7
Processed sample 8
Processed sample 9
Processed sample 10
Processed sample 11
Processed sample 12
Processed sample 13
Processed sample 14
Processed sample 15
Processed sample 16
Processed sample 17
Processed sample 18
Processed sample 19
Processed sample 20
Processed sample 21
Processed sample 22
Processed sample 23
Processed sample 24
Processed sample 25
Processed sample 26
Processed sample 27
Processed sample 28
Processed sample 29
Processed sample 30
Processed sample 31
Processed sample 32
Processed sample 33
Processed sample 34
Processed sample 35
Processed sample 36
Processed sample 37
Processed sample 38
Processed sample 39
Processed sample 40
Processed sample 41
Processed sample 42
Processed sample 43
Processed sample 44
Processed sample 45
Processed sample 46
Processed sample 47
Processed sample 48
Processed sample 49
Processed 

#### Saving results

In [24]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [25]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(500, 33)

In [26]:
output_dir = f"{params['output_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
create_folder(output_dir)
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

#### Loss Retrieval

In [27]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_dir = f"{params['callbacks_path']}/{params['dataset']['current']}/{params['current_model']}_q_{params['quantization']}"
    output_loss = []
    for current_batch in range(size):
        out = np.load(f"{output_dir}/_loss_batch[{current_batch}].npy")
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [28]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [29]:
output_loss

[1.7954045534133911,
 1.1617348194122314,
 1.7954045534133911,
 0.9020307064056396,
 0.9918232560157776,
 1.1845952272415161,
 1.1494802236557007,
 0.469643235206604,
 1.831680178642273,
 1.251354694366455,
 0.8918948173522949,
 0.5787695050239563,
 0.6802713871002197,
 0.6912832260131836,
 1.085374116897583,
 1.5927038192749023,
 1.0017364025115967,
 0.7691758275032043,
 0.8618466258049011,
 1.2188458442687988,
 0.9273914098739624,
 1.1626330614089966,
 1.0181013345718384,
 1.3467353582382202,
 1.043167233467102,
 0.9726211428642273,
 1.0260584354400635,
 1.2593399286270142,
 0.7125228643417358,
 1.7080647945404053,
 1.0665740966796875,
 1.2389169931411743,
 0.871314525604248,
 1.1254997253417969,
 0.959561824798584,
 0.6129717826843262,
 1.606524109840393,
 0.5073738098144531,
 1.0235531330108643,
 0.8279467225074768,
 0.8385608792304993,
 1.5581039190292358,
 0.6261972784996033,
 1.6894689798355103,
 2.1556308269500732,
 0.9809377193450928,
 0.6843172311782837,
 0.7465140223503113,


In [30]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,input_lenght,input_ids,max_prob,min_prob,actual_prob,loss
0,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,def get_uploaded_datasets(self):\n if l...,https://github.com/OpenMined/PySyft.git,Python,...,9,100,"raise Exception(""User cannot connect to this d...",Warning,360,"[822, 679, 29918, 9009, 287, 29918, 14538, 169...","[(<PRE>, 0.7492900490760803), (module, 0.47943...","[(<s>, 6.321457571810407e-13), ($}, 1.21013518...","[(def, 0.0007611316395923495), (get, 0.0267730...",1.795405
1,115917,6eb408a9973fbc24c973d6524dc34cb9b1e0ee05,mindsdb,mindsdb/api/mongo/responders/delete.py,delete.py,_result,del model interface,"def _result(self, query, request_env, mindsdb_...",https://github.com/mindsdb/mindsdb.git,Python,...,10,129,"raise Exception(""For db.predictors.delete oper...",Warning,292,"[822, 903, 2914, 29898, 1311, 29892, 2346, 298...","[(<PRE>, 0.7492886185646057), (module, 0.47944...","[(<s>, 6.321602312800434e-13), ($}, 1.21012075...","[(def, 0.0007611338514834642), (_, 0.009010342...",1.161735
2,2758,0e9baa6d2c7a752eb32e6da2359470f95a3efeec,PySyft,packages/syft/src/syft/oblv/model.py,model.py,get_uploaded_datasets,Adding method to get datasets list,def get_uploaded_datasets(self):\n if l...,https://github.com/OpenMined/PySyft.git,Python,...,6,190,"raise Exception(""Either proxy not running or n...",Warning,360,"[822, 679, 29918, 9009, 287, 29918, 14538, 169...","[(<PRE>, 0.7492900490760803), (module, 0.47943...","[(<s>, 6.321457571810407e-13), ($}, 1.21013518...","[(def, 0.0007611316395923495), (get, 0.0267730...",1.795405
3,116232,b734716c29a8b7613354e160ea1a3aea111f0831,mindsdb,mindsdb/interfaces/database/integrations.py,integrations.py,get_handler,Improve handlers files storage,"def get_handler(self, name, company_id=None, c...",https://github.com/mindsdb/mindsdb.git,Python,...,15,97,"raise Exception(f""Cant find handler for '{inte...",Warning,535,"[822, 679, 29918, 13789, 29898, 1311, 29892, 1...","[(<PRE>, 0.7492900490760803), (module, 0.47943...","[(<s>, 6.321457571810407e-13), ($}, 1.21013518...","[(def, 0.0007611316395923495), (get, 0.0267730...",0.902031
4,115111,d725c063e3ac3ab919fc8ed5f969fef8bc478209,mindsdb,mindsdb/api/mysql/mysql_proxy/datahub/datanode...,integration_datanode.py,select,fix,"def select(self, query):\n result = sel...",https://github.com/mindsdb/mindsdb.git,Python,...,4,49,raise Exception(result.error_message),Warning,147,"[822, 1831, 29898, 1311, 29892, 2346, 1125, 13...","[(<PRE>, 0.7492900490760803), (module, 0.47944...","[(<s>, 6.321939499676077e-13), ($}, 1.21012130...","[(def, 0.0007611415348947048), (select, 0.0004...",0.991823


In [31]:
## Saving CheckPoint 2
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

In [32]:
print("================================= PROCESS COMPLETED =================================")

================================= PROCESS COMPLETED =================================


In [33]:
del model
torch.cuda.empty_cache()
gc.collect()

130